#Imports

In [ ]:
!pip install woodelf_explainer==0.2.12

In [ ]:
import xgboost as xgb
import pandas as pd
import numpy as np
from typing import Union, Dict, Optional, Tuple, Set, List
from math import factorial
import time
from copy import copy
from tqdm import tqdm
from collections import defaultdict
from sklearn.metrics import accuracy_score, f1_score
import scipy
import shap

import lightgbm as lgb

# For GPU execution
# import cupy as cp

In [ ]:
from woodelf.parse_models import load_decision_tree_ensemble_model
from woodelf.cube_metric import CubeMetric, ShapleyInteractionValues, ShapleyValues

import woodelf


In [ ]:
# Useful if you run this on google colab and downloaded the data into your drive.
# If you run the notebook in other environment remove these lines and change the 'pd.read_csv()' function in this notebook to read from
# where you saved you data
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## Environment Note
All the code below, that works with the EEEI-CIS fraud data will run perfectly in the free version of Google Colab.
The free version machine have 12GB RAM and this is enough for this part of the notebook.

Next, we will load the KDD-Cup dataset. As it includes millions of rows we need more than 12GB to run the algorithm.
This part of the notebook needs the 50GB CPU machine. It is available for subscribed members for Google Colab (toggle the High-RAM option in Runtime->Change runtime type button).

If you only bought computation units, but not a subscription, you can still run the full notebook using the v2-8-TPU machine. This machine have more then enough RAM and it is pretty cheap. Its CPU is a bit slower though, so you will get a slower runtimes.

Final note: preformance in Colab depends on the machine allocated (the CPU option can allocate several types of machines) and on load balancing inside Colab (when more users use the system running times might be slower). So the running times of our algorithms and the shap python package can very. The difference between our approach and the current state-of-the-art is big enough to be noticed, but the excat running times can be slightly different from the ones stated our paper.

# Fraud Data Preprocessing and Model training

In [ ]:
transactions_train = pd.read_parquet('drive/MyDrive/CURRENT_DIR/data/ieee_cis_fraud_train.parquet') # columns are train_features + ['isFraud']
transactions_test = pd.read_parquet('drive/MyDrive/CURRENT_DIR/data/ieee_cis_fraud_test.parquet') # columns are train_features + ['isFraud']

train_features = [f for f in transactions_train.columns if f != 'isFraud']
fraud_train = transactions_train[train_features]
fraud_test = transactions_test[train_features]

In [ ]:
LIGHTGBM_PARAMS = {
    "boosting_type": "gbdt",
    "objective": "binary",
    "metric": "auc",
    "learning_rate": 0.1,

    # Allow high depth and enough leaves to reach this possible depth
    "num_leaves": 2024,
    "max_depth": 10,
    "min_data_in_leaf": 500,        # Does provide some regulation

    # Sampling (stability + reduces overfit)
    "feature_fraction": 0.8,
    "bagging_fraction": 0.8,
    "bagging_freq": 1,

    # Practical
    "verbosity": -1,
    "seed": 42,
    "force_col_wise": True,            # often faster/safer for wide data
}


def print_model_stat(model, features):
    mobj = load_decision_tree_ensemble_model(model, features)
    depths = {}
    for tree in mobj.trees:
        for leaf, path in tree.get_all_leaves_with_paths():
            depth = len(path)
            if depth not in depths:
                depths[depth] = 0
            depths[depth] += 1

    for depth in sorted(list(depths.keys())):
        print(f"\tPaths of depth {depth}: {depths.get(depth, 0)} paths")

def lightgbm_model(X_train, y_train, params, num_rounds=100):
    train_set = lgb.Dataset(X_train, label=y_train, free_raw_data=False)
    return lgb.train(
        params=params,
        train_set=train_set,
        num_boost_round=num_rounds
    )


def different_depth_lightgbm_models(trainset, y, params, num_rounds, depths):
    models = {}
    for depth in depths:
        new_params = params.copy()
        new_params['max_depth'] = depth
        models[depth] = lightgbm_model(trainset, y, new_params, num_rounds=num_rounds)
        print("\n\n")
        print(f"Trained on depth {depth}")
        print_model_stat(models[depth], list(trainset.columns))
    return models

In [ ]:
gbm_100trees = different_depth_lightgbm_models(transactions_train[train_features], transactions_train['isFraud'], LIGHTGBM_PARAMS, num_rounds=100, depths=[6,9,12,15,18,21])




Trained on depth 6
	Paths of depth 1: 6 paths
	Paths of depth 2: 60 paths
	Paths of depth 3: 135 paths
	Paths of depth 4: 284 paths
	Paths of depth 5: 473 paths
	Paths of depth 6: 2086 paths



Trained on depth 9
	Paths of depth 1: 16 paths
	Paths of depth 2: 55 paths
	Paths of depth 3: 138 paths
	Paths of depth 4: 258 paths
	Paths of depth 5: 454 paths
	Paths of depth 6: 710 paths
	Paths of depth 7: 959 paths
	Paths of depth 8: 1222 paths
	Paths of depth 9: 3752 paths



Trained on depth 12
	Paths of depth 1: 15 paths
	Paths of depth 2: 55 paths
	Paths of depth 3: 143 paths
	Paths of depth 4: 290 paths
	Paths of depth 5: 472 paths
	Paths of depth 6: 636 paths
	Paths of depth 7: 884 paths
	Paths of depth 8: 1140 paths
	Paths of depth 9: 1496 paths
	Paths of depth 10: 1681 paths
	Paths of depth 11: 1991 paths
	Paths of depth 12: 4782 paths



Trained on depth 15
	Paths of depth 1: 16 paths
	Paths of depth 2: 72 paths
	Paths of depth 3: 121 paths
	Paths of depth 4: 278 paths
	Paths of

In [ ]:
gbm_10trees = different_depth_lightgbm_models(transactions_train[train_features], transactions_train['isFraud'], LIGHTGBM_PARAMS, num_rounds=10, depths=[12,15,18, 21])




Trained on depth 12
	Paths of depth 3: 6 paths
	Paths of depth 4: 54 paths
	Paths of depth 5: 63 paths
	Paths of depth 6: 89 paths
	Paths of depth 7: 137 paths
	Paths of depth 8: 158 paths
	Paths of depth 9: 191 paths
	Paths of depth 10: 204 paths
	Paths of depth 11: 248 paths
	Paths of depth 12: 552 paths



Trained on depth 15
	Paths of depth 2: 1 paths
	Paths of depth 3: 5 paths
	Paths of depth 4: 52 paths
	Paths of depth 5: 61 paths
	Paths of depth 6: 85 paths
	Paths of depth 7: 158 paths
	Paths of depth 8: 141 paths
	Paths of depth 9: 192 paths
	Paths of depth 10: 215 paths
	Paths of depth 11: 269 paths
	Paths of depth 12: 265 paths
	Paths of depth 13: 290 paths
	Paths of depth 14: 319 paths
	Paths of depth 15: 642 paths



Trained on depth 18
	Paths of depth 2: 1 paths
	Paths of depth 3: 7 paths
	Paths of depth 4: 50 paths
	Paths of depth 5: 63 paths
	Paths of depth 6: 74 paths
	Paths of depth 7: 151 paths
	Paths of depth 8: 161 paths
	Paths of depth 9: 176 paths
	Paths of dep

In [ ]:
gbm_1trees = different_depth_lightgbm_models(transactions_train[train_features], transactions_train['isFraud'], LIGHTGBM_PARAMS, num_rounds=1, depths=[12,15,18, 21])




Trained on depth 12
	Paths of depth 3: 3 paths
	Paths of depth 4: 8 paths
	Paths of depth 6: 2 paths
	Paths of depth 7: 3 paths
	Paths of depth 8: 6 paths
	Paths of depth 9: 10 paths
	Paths of depth 10: 11 paths
	Paths of depth 11: 19 paths
	Paths of depth 12: 30 paths



Trained on depth 15
	Paths of depth 3: 3 paths
	Paths of depth 4: 8 paths
	Paths of depth 6: 2 paths
	Paths of depth 7: 3 paths
	Paths of depth 8: 6 paths
	Paths of depth 9: 10 paths
	Paths of depth 10: 11 paths
	Paths of depth 11: 19 paths
	Paths of depth 12: 13 paths
	Paths of depth 13: 17 paths
	Paths of depth 14: 19 paths
	Paths of depth 15: 30 paths



Trained on depth 18
	Paths of depth 3: 3 paths
	Paths of depth 4: 8 paths
	Paths of depth 6: 2 paths
	Paths of depth 7: 3 paths
	Paths of depth 8: 6 paths
	Paths of depth 9: 10 paths
	Paths of depth 10: 11 paths
	Paths of depth 11: 19 paths
	Paths of depth 12: 13 paths
	Paths of depth 13: 17 paths
	Paths of depth 14: 19 paths
	Paths of depth 15: 11 paths
	Paths 

In [ ]:
def high_depth_woodelf_on_models_dict(
        models, consumer_data: pd.DataFrame, background_data: pd.DataFrame, metric: CubeMetric, global_importance: bool = False, GPU=False
    ):
    running_times = {}
    for key, model in models.items():
        start_time = time.time()
        woodelf.high_depth_woodelf.woodelf_for_high_depth(model, consumer_data, background_data, metric=metric, global_importance=global_importance)
        running_time = time.time() - start_time
        running_times[key] = running_time
        print(f"On Depth {key} Took: {running_time}")
    return running_times

In [ ]:
def simple_woodelf_on_models_dict(
        models, consumer_data: pd.DataFrame, background_data: pd.DataFrame, metric: CubeMetric, global_importance: bool = False, GPU=False,
    ):
    running_times = {}
    for key, model in models.items():
        start_time = time.time()
        woodelf.simple_woodelf.calculate_background_metric(model, consumer_data, background_data, metric=metric, global_importance=global_importance)
        running_time = time.time() - start_time
        running_times[key] = running_time
        print(f"On Depth {key} Took: {running_time}")
    return running_times

In [ ]:
def shap_on_models_dict(models, consumer_data: pd.DataFrame, background_data: pd.DataFrame):
    running_times = {}
    for key, model in models.items():
        start_time = time.time()
        explainer = shap.TreeExplainer(model, background_data, feature_perturbation='interventional')
        simple_shap_values = explainer.shap_values(consumer_data)
        running_time = time.time() - start_time
        running_times[key] = running_time
        print(f"On Depth {key} Took: {running_time}")
    return running_times

# Background SHAP

In [ ]:
woodelf_running_times = high_depth_woodelf_on_models_dict(
    {6: gbm_100trees[6], 9: gbm_100trees[9], 12: gbm_100trees[12], 15: gbm_100trees[15], 18: gbm_100trees[18], 21: gbm_100trees[21]},
    consumer_data=transactions_test[train_features], background_data=transactions_train[train_features], metric=ShapleyValues(),
    global_importance = False, GPU=False
)
print("Background SHAP: " + " & ".join([str(woodelf_running_times[d]) for d in sorted(list(woodelf_running_times.keys()))]))

Preprocessing the trees and computing SHAP: 100%|██████████| 100/100 [00:13<00:00,  7.38it/s]


M time: 0.0 sec, s time: 0.44 sec (f prepare time: 0.14769673347473145)
On Depth 6 Took: 13.660076379776001


Preprocessing the trees and computing SHAP: 100%|██████████| 100/100 [00:46<00:00,  2.14it/s]


M time: 0.01 sec, s time: 1.97 sec (f prepare time: 0.5643846988677979)
On Depth 9 Took: 47.275256395339966


Preprocessing the trees and computing SHAP: 100%|██████████| 100/100 [01:42<00:00,  1.03s/it]


...

M time: 1.2 sec, s time: 88.03 sec (f prepare time: 6.923884630203247)
On Depth 15 Took: 285.0737280845642


Preprocessing the trees and computing SHAP: 100%|██████████| 100/100 [23:49<00:00, 14.30s/it]


M time: 11.69 sec, s time: 1063.14 sec (f prepare time: 44.28532028198242)
On Depth 18 Took: 1442.5526540279388


Preprocessing the trees and computing SHAP: 100%|██████████| 100/100 [2:58:15<00:00, 106.96s/it]

M time: 108.34 sec, s time: 9776.61 sec (f prepare time: 409.85093569755554)
On Depth 21 Took: 10805.23444199562
Background SHAP: 13.660076379776001 & 47.275256395339966 & 103.52650380134583 & 285.0737280845642 & 1442.5526540279388 & 10805.23444199562


In [ ]:
woodelf_running_times = simple_woodelf_on_models_dict(
    {6: gbm_100trees[6], 9: gbm_100trees[9], 12: gbm_100trees[12], 15: gbm_10trees[15]}, # Depth 18 crashes due to RAM +  RAM crash a tree 3 of depth 15
    consumer_data=transactions_test[train_features], background_data=transactions_train[train_features], metric=ShapleyValues(),
    global_importance = False, GPU=False
)
print("Background SHAP: " + " & ".join([str(woodelf_running_times[d]) for d in sorted(list(woodelf_running_times.keys()))]))

Preprocessing the trees: 100%|██████████| 100/100 [00:05<00:00, 16.77it/s]


cache misses: 71, cache used: 2973, M computation time: 0.16 sec, s computation time: 0.16 sec


Computing the values: 100%|██████████| 100/100 [00:06<00:00, 15.61it/s]


On Depth 6 Took: 12.492154359817505


Preprocessing the trees: 100%|██████████| 100/100 [00:59<00:00,  1.67it/s]


...

Preprocessing the trees: 100%|██████████| 100/100 [56:13<00:00, 33.74s/it]


cache misses: 1503, cache used: 12082, M computation time: 3250.75 sec, s computation time: 33.51 sec


Computing the values: 100%|██████████| 100/100 [01:25<00:00,  1.17it/s]


On Depth 12 Took: 3459.8345334529877


Preprocessing the trees:  20%|██        | 2/10 [48:29<3:18:22, 1487.83s/it]

In [ ]:
woodelf_running_times = simple_woodelf_on_models_dict(
    {15: gbm_1trees[15]}, # Depth 18 crashes due to RAM +  RAM crash a tree 3 of depth 15
    consumer_data=transactions_test[train_features], background_data=transactions_train[train_features], metric=ShapleyValues(),
    global_importance = False, GPU=False
)
print("Background SHAP: " + " & ".join([str(woodelf_running_times[d]) for d in sorted(list(woodelf_running_times.keys()))]))

Preprocessing the trees: 100%|██████████| 1/1 [21:22<00:00, 1282.00s/it]


cache misses: 29, cache used: 112, M computation time: 1250.09 sec, s computation time: 10.0 sec


Computing the values: 100%|██████████| 1/1 [00:03<00:00,  3.55s/it]

On Depth 15 Took: 1285.614224910736
Background SHAP: 1285.614224910736


In [ ]:
train_sample = fraud_train.sample(10, random_state=42)

shap_running_times = shap_on_models_dict(
    gbm_100trees, consumer_data=fraud_test, background_data=train_sample,
)
print("Background SHAP: " + " & ".join([str(shap_running_times[d]) for d in sorted(list(shap_running_times.keys()))]))

100%|===================| 117981/118108 [00:58<00:00]       

On Depth 6 Took: 61.20456600189209


 99%|===================| 117232/118108 [01:49<00:00]       

On Depth 9 Took: 111.20987391471863


100%|===================| 118041/118108 [02:52<00:00]       

On Depth 12 Took: 174.71757745742798


100%|===================| 117795/118108 [03:59<00:00]       

On Depth 15 Took: 242.83412337303162


100%|===================| 117984/118108 [05:10<00:00]       

On Depth 18 Took: 314.1569471359253
Background SHAP: 61.20456600189209 & 111.20987391471863 & 174.71757745742798 & 242.83412337303162 & 314.1569471359253


# Background SHAP IV

In [ ]:
woodelf_running_times = high_depth_woodelf_on_models_dict(
    {6: gbm_100trees[6], 9: gbm_100trees[9], 12: gbm_100trees[12], 15: gbm_100trees[15], 18: gbm_100trees[18]}, # depth 21 crashes due to RAM
    consumer_data=transactions_test[train_features].sample(10_000, random_state=42), background_data=transactions_train[train_features], metric=ShapleyInteractionValues(),
    global_importance = False, GPU=False
)
print("Background SHAP: " + " & ".join([str(woodelf_running_times[d]) for d in sorted(list(woodelf_running_times.keys()))]))

Preprocessing the trees and computing SHAP: 100%|██████████| 100/100 [00:09<00:00, 11.09it/s]


M time: 0.0 sec, s time: 0.57 sec (f prepare time: 0.1541919708251953)
On Depth 6 Took: 9.148506879806519


Preprocessing the trees and computing SHAP: 100%|██████████| 100/100 [00:31<00:00,  3.13it/s]


M time: 0.02 sec, s time: 3.26 sec (f prepare time: 0.5597352981567383)
On Depth 9 Took: 32.32128024101257


Preprocessing the trees and computing SHAP: 100%|██████████| 100/100 [01:50<00:00,  1.10s/it]


M time: 0.32 sec, s time: 40.47 sec (f prepare time: 1.78265380859375)
On Depth 12 Took: 111.01148343086243


Preprocessing the trees and computing SHAP: 100%|██████████| 100/100 [16:27<00:00,  9.88s/it]


M time: 4.21 sec, s time: 792.83 sec (f prepare time: 7.0785441398620605)
On Depth 15 Took: 992.6146140098572


Preprocessing the trees and computing SHAP: 100%|██████████| 100/100 [2:34:23<00:00, 92.64s/it]

M time: 55.64 sec, s time: 8165.65 sec (f prepare time: 46.38955807685852)
On Depth 18 Took: 9320.691369771957
Background SHAP: 9.148506879806519 & 32.32128024101257 & 111.01148343086243 & 992.6146140098572 & 9320.691369771957


In [ ]:
woodelf_running_times = simple_woodelf_on_models_dict(
    {6: gbm_100trees[6], 9: gbm_100trees[9], 12: gbm_100trees[12]}, # depth 15 crashes due to RAM
    consumer_data=transactions_test[train_features].sample(10_000, random_state=42), background_data=transactions_train[train_features], metric=ShapleyInteractionValues(),
    global_importance = False, GPU=False
)
print("Background SHAP: " + " & ".join([str(woodelf_running_times[d]) for d in sorted(list(woodelf_running_times.keys()))]))

Preprocessing the trees: 100%|██████████| 100/100 [00:06<00:00, 15.26it/s]


cache misses: 71, cache used: 2973, M computation time: 0.27 sec, s computation time: 0.35 sec


Computing the values: 100%|██████████| 100/100 [00:01<00:00, 50.74it/s]


On Depth 6 Took: 8.685823440551758


Preprocessing the trees: 100%|██████████| 100/100 [01:31<00:00,  1.09it/s]


...

On Depth 9 Took: 107.80704307556152


Preprocessing the trees: 100%|██████████| 100/100 [2:08:01<00:00, 76.82s/it]


cache misses: 1503, cache used: 12082, M computation time: 7456.4 sec, s computation time: 129.31 sec


Computing the values: 100%|██████████| 100/100 [03:19<00:00,  2.00s/it]


On Depth 12 Took: 7883.167765617371
Background SHAP: 8.685823440551758 & 107.80704307556152 & 7883.167765617371


In [ ]:
# shap Package does not support Background SHAP IV

# RAM Crashes

In [ ]:
# Crashed due to RAM

woodelf_running_times = simple_woodelf_on_models_dict(
    {18: gbm_1trees[18]},
    consumer_data=transactions_test[train_features], background_data=transactions_train[train_features], metric=ShapleyValues(),
    global_importance = False, GPU=False
)
print("Background SHAP: " + " & ".join([str(woodelf_running_times[d]) for d in sorted(list(woodelf_running_times.keys()))]))

Preprocessing the trees:   0%|          | 0/1 [00:00<?, ?it/s]

In [ ]:
# Crashed due to RAM

woodelf_running_times = simple_woodelf_on_models_dict(
    {15: gbm_1trees[15]},
    consumer_data=transactions_test[train_features], background_data=transactions_train[train_features], metric=ShapleyInteractionValues(),
    global_importance = False, GPU=False
)
print("Background SHAP: " + " & ".join([str(woodelf_running_times[d]) for d in sorted(list(woodelf_running_times.keys()))]))

Preprocessing the trees:   0%|          | 0/1 [00:00<?, ?it/s]

In [ ]:
# Crashed due to RAM (in the HighDepthPathToMatrices.build_matrices phase)

woodelf_running_times = high_depth_woodelf_on_models_dict(
    {21: gbm_1trees[21]},
    consumer_data=transactions_test[train_features].sample(10_000, random_state=42), background_data=transactions_train[train_features], metric=ShapleyInteractionValues(),
    global_importance = False, GPU=False
)
print("Background SHAP: " + " & ".join([str(woodelf_running_times[d]) for d in sorted(list(woodelf_running_times.keys()))]))